# Cross-Session Comparison

Use this notebook when you have multiple research sessions and want to answer:
- Which strategy has the best risk-adjusted returns?
- Are they correlated? Can we run them together?
- Which strategy works in which market regime?
- What's the portfolio Sharpe if we combine them?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from gnomepy_research import compare_results, load_results
from gnomepy_research.analysis.portfolio import (
    compute_cross_strategy_correlation,
    combined_sharpe,
    plot_correlation_matrix,
)
from gnomepy_research.reporting.backtest import detect_regimes, pnl_by_regime
from gnomepy.reporting.metrics import compute_sharpe

## Configuration

Add paths to any best iteration from each session you want to compare.

In [ ]:
SESSIONS = {
    "n_exchange_arb":    "gnomepy_research/sessions/n_exchange_arb/results/iter_015",
    "hyperliquid_btc_mm": "gnomepy_research/sessions/hyperliquid_btc_mm/results/iter_023",
}

## 1. Summary Comparison Table

In [ ]:
comparison = compare_results(*SESSIONS.values(), metric="sharpe")
comparison.insert(0, "session", list(SESSIONS.keys()))
comparison

## 2. PnL Curves Overlay

Normalized to $1 of starting capital so curves are directly comparable.

In [ ]:
reports = {name: load_results(path) for name, path in SESSIONS.items()}

fig = go.Figure()
for name, report in reports.items():
    pnl = report.pnl_curve
    if pnl.empty:
        continue
    # Normalize: show PnL as a fraction of max absolute PnL for visual comparison
    scale = abs(pnl).max() or 1.0
    normed = pnl / scale
    # Downsample
    step = max(1, len(normed) // 5000)
    normed = normed.iloc[::step]
    fig.add_trace(go.Scatter(x=normed.index, y=normed.values, name=name, line=dict(width=1.0)))

fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.5)
fig.update_layout(
    title="PnL Curves (normalized)",
    height=400,
    hovermode="x unified",
    yaxis_title="Normalized PnL",
)
fig.show()

## 3. Correlation Analysis

Low correlation between strategies is valuable — it means combining them
improves the portfolio Sharpe more than a single strategy ever could.

In [ ]:
pnl_curves = {name: report.pnl_curve for name, report in reports.items()}

# Resample to 1-minute returns for correlation
rets = {}
for name, pnl in pnl_curves.items():
    if not pnl.empty:
        rets[name] = pnl.resample("1min").last().diff().dropna()

if len(rets) > 1:
    rets_df = pd.DataFrame(rets).dropna()
    corr = rets_df.corr()
    print("Pairwise PnL return correlation:")
    print(corr.round(3))
    plot_correlation_matrix(corr).show()
else:
    print("Need at least 2 sessions to compute correlation.")

## 4. Portfolio Construction

What's the combined Sharpe if we run these strategies together?
Equal weighting vs. Sharpe-proportional weighting.

In [ ]:
if len(pnl_curves) > 1:
    names = list(pnl_curves.keys())
    curves = list(pnl_curves.values())

    # Equal weights
    equal_weights = [1.0 / len(curves)] * len(curves)
    eq_sharpe = combined_sharpe(curves, equal_weights)
    print(f"Equal-weight portfolio Sharpe: {eq_sharpe:.3f}")

    # Individual Sharpes for comparison
    print("\nIndividual Sharpes:")
    for name, pnl in pnl_curves.items():
        if not pnl.empty:
            metrics = compute_sharpe(pnl)
            print(f"  {name}: {metrics.get('sharpe', 0):.3f}")

## 5. Per-Regime Comparison

Which strategy performs well in which market regime?
Uses the market data from the first session as the regime reference.

In [ ]:
# Use market data from the first available report for regime labels
first_report = next(iter(reports.values()))
market_df = first_report._market_df

if not market_df.empty:
    regimes = detect_regimes(market_df)

    regime_rows = []
    for name, report in reports.items():
        pnl = report.pnl_curve
        if pnl.empty:
            continue
        breakdown = pnl_by_regime(pnl, regimes)
        for regime, stats in breakdown.items():
            regime_rows.append({
                "session": name,
                "regime": regime,
                "final_pnl": stats.get("final_pnl", 0),
                "sharpe": stats.get("sharpe", 0),
                "pct_time": stats.get("pct_time", 0),
            })

    regime_df = pd.DataFrame(regime_rows)
    print("PnL by regime and session:")
    pivot = regime_df.pivot(index="regime", columns="session", values="final_pnl").round(4)
    print(pivot)

    px.bar(
        regime_df,
        x="regime",
        y="sharpe",
        color="session",
        barmode="group",
        title="Sharpe by Market Regime",
    ).show()
else:
    print("No market data available for regime detection.")

## 6. Sweep Comparison (Optional)

If you ran a parameter sweep and want to compare all jobs in one table,
point `compare_results` at all the job subdirectories.

In [ ]:
import glob

# Example: compare all iterations of a single session
# Uncomment and adjust the path:

# SESSION_DIR = "gnomepy_research/sessions/n_exchange_arb/results"
# iter_paths = sorted(glob.glob(f"{SESSION_DIR}/iter_*"))
# sweep_df = compare_results(*iter_paths, metric="sharpe")
# sweep_df.insert(0, "iteration", [p.split("/")[-1] for p in iter_paths])
# sweep_df